# 第11回：データから学習して評価する

アヤメの計測値から3種類を分類します．コードを上から順に実行し，学習・検証・テストの役割を観察します．

**先に読むこと**：設定の変更は検証データで比較します．テストは条件を決めた後の最終確認に使います．テストを見て設定を戻すと，独立した評価ではなくなります．

Google Colabの「ノートブックをアップロード」またはJupyterで開いてください．データはscikit-learnに含まれ，別のダウンロードは不要です．個人情報は使いません．

In [ ]:
# Google Colabでは通常，このライブラリが用意されています．
# ローカル環境で不足している場合だけ，必要なライブラリを入れます．
import importlib.util, subprocess, sys
packages = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib', 'sklearn':'scikit-learn'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
print('Python:', sys.version.split()[0], 'scikit-learn:', sklearn.__version__)


## 1．設定を決める

最初はそのまま実行します．比較実験では `FEATURES` を `[0, 1]` にする，または `TREE_DEPTH` を変えるなど，**一つだけ変更**します．変更後はここから下を再実行し，検証結果を記録します．

In [ ]:
SEED = 42
FEATURES = [0, 1, 2, 3]
TREE_DEPTH = 3
from sklearn.datasets import load_iris
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
iris = load_iris()
X = iris.data[:, FEATURES]
y = iris.target
names = iris.target_names
feature_names = [iris.feature_names[i] for i in FEATURES]
print('データの形:', X.shape, 'クラス:', list(names))
print(pd.DataFrame(X, columns=feature_names).head())
print('クラスごとの件数:', np.bincount(y))


## 2．データを観察する

行は一つの花，列は計測値です．散布図で，種類の重なりと区別しやすい特徴を見ます．クラス名は正解ラベルなので，入力に混ぜません．

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
for k, name in enumerate(names):
    ax.scatter(X[y==k, 0], X[y==k, 1], label=name, alpha=.7)
ax.set(xlabel=feature_names[0], ylabel=feature_names[1])
ax.legend(); plt.show()


## 3．訓練90件・検証30件・テスト30件へ分ける

最初にテスト20％を取り分け，残り120件の25％を検証へ分けます．`stratify` でクラスの割合をそろえます．同じ花が複数の分割へ入らないよう，行番号も確認します．実際の業務では時刻・人物・店舗などに合わせた分割が必要です．

In [ ]:
all_idx = np.arange(len(y))
trainval_idx, test_idx = train_test_split(all_idx, test_size=.2, stratify=y, random_state=SEED)
train_idx, val_idx = train_test_split(trainval_idx, test_size=.25, stratify=y[trainval_idx], random_state=SEED)
assert not (set(train_idx)&set(val_idx) or set(train_idx)&set(test_idx) or set(val_idx)&set(test_idx))
X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
print('訓練・検証・テスト:', len(train_idx), len(val_idx), len(test_idx))


## 4．同じ条件でモデルを比較する

標準化はロジスティック回帰のPipeline内に置き，訓練データだけから平均・標準偏差を求めます．決定木はここでは標準化しません．常に一つのクラスを出すDummyClassifierを基準にします．

`fit` は学習，`predict` は学習したモデルの推論です．

In [ ]:
models = {
    'baseline': DummyClassifier(strategy='most_frequent'),
    'logistic': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEED)),
    'tree': DecisionTreeClassifier(max_depth=TREE_DEPTH, random_state=SEED),
}
records = []
for name, model in models.items():
    model.fit(X_train, y_train)
    records.append({'model':name, 'train_accuracy':accuracy_score(y_train, model.predict(X_train)),
                    'validation_accuracy':accuracy_score(y_val, model.predict(X_val))})
comparison = pd.DataFrame(records)
print(comparison.to_string(index=False))


## 5．検証データの誤りを読む

検証正解率が最も高いモデルを一つ選びます．同点なら辞書で先に記載したモデルを選ぶ規則とします．混同行列は行＝正解，列＝予測です．誤りが0件でも，今の30件だけの結果であることに注意します．

In [ ]:
best_name = comparison.loc[comparison.validation_accuracy.idxmax(), 'model']
selected = models[best_name]
val_pred = selected.predict(X_val)
print('選んだモデル:', best_name)
ConfusionMatrixDisplay.from_predictions(y_val, val_pred, labels=np.arange(3), display_labels=names)
plt.show()
wrong = np.flatnonzero(val_pred != y_val)
if len(wrong):
    error_table = pd.DataFrame(X_val[wrong], columns=feature_names)
    error_table['true'] = names[y_val[wrong]]
    error_table['predicted'] = names[val_pred[wrong]]
    print(error_table.to_string(index=False))
else:
    print('この検証データでは誤分類0件．別のデータでの正しさは保証しません．')


## 6．比較実験（ここで一度止まる）

1. 使う特徴量，または木の深さを一つだけ変えます．
2. 同じ分割の検証結果を記録し，元の設定と比べます．
3. 条件を決め，次のテストを実行します．

何度も検証して最適化すると検証データにも合わせすぎるため，試す条件を授業内で少数に決めます．「ランタイム→すべてのセルを実行」では最初の設定でテストまで進むので，授業ではセルごとに実行してください．

## 7．条件を決めた後のテスト

モデルと設定を固定したら，選んだ方式を訓練＋検証120件で学習し直し，取り分けていた30件で一度評価します．ここから設定を変更した場合は，新しい独立したテストが必要です．

In [ ]:
from sklearn.base import clone
final_model = clone(selected).fit(X[trainval_idx], y[trainval_idx])
test_pred = final_model.predict(X[test_idx])
test_accuracy = accuracy_score(y[test_idx], test_pred)
print('最終テストの正解率:', round(test_accuracy, 4))
print('テスト件数:', len(test_idx), '正解件数:', int((test_pred==y[test_idx]).sum()))
ConfusionMatrixDisplay.from_predictions(y[test_idx], test_pred, labels=np.arange(3), display_labels=names)
plt.show()


## 8．提出する短い記録

- どの特徴量とモデルを使ったか．
- 基準・訓練・検証・テストの結果と件数．
- 変更した条件と，検証結果の違い．
- 間違いやすい種類，この評価でまだ分からないこと．

環境やライブラリの版が違うと数値が多少変わることがあります．他の人より正解率が高いことだけで評価しません．

In [ ]:
print({'seed':SEED, 'features':FEATURES, 'tree_depth':TREE_DEPTH, 'selected_model':best_name,
       'n_train':len(train_idx), 'n_validation':len(val_idx), 'n_test':len(test_idx),
       'test_accuracy':float(test_accuracy), 'sklearn_version':sklearn.__version__})


## 参考資料

- [Iris dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html)
- [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)
- [Common pitfalls](https://scikit-learn.org/stable/common_pitfalls.html)

説明とコードは授業用に作成したものです．データセットの由来はscikit-learnの資料を参照してください．